# Exercise — Fix the Impurities, Don't Drop Them

`loan_prediction_dataset.csv` (same file as the main notebook) arrives with real messiness:
missing values in several columns, and inconsistent text casing (`"yes"` vs `"Yes"`, `"urban"` vs `"Urban"`).

The lazy fix is `df.dropna()`. **Don't do that.** In this exercise you'll prove *why* it's lazy,
then build the correct pipeline: fix casing, impute properly (train-only, no leakage), and confirm
you kept every row.

### Rules
- No `dropna()` anywhere in your final pipeline.
- Anything you fit (imputers, encoders) must be fit on the **train** split only.
- By the end, `X_train`/`X_test` should have **zero** missing values and the **same row counts**
  as right after the split.


### Step 1. Import `numpy`, `pandas`, `train_test_split`, `SimpleImputer`, and `DecisionTreeClassifier` + `accuracy_score`.

### Step 2. Load `loan_prediction_dataset.csv` into `df`. Print its shape, and print `df.isna().sum()` to see exactly which columns are dirty.

### Step 3. The "just drop it" gut-check

Before fixing anything, measure the cost of the lazy approach:
- Compute `dropped = df.dropna()`. Print how many rows survive, and what **percentage** of the
  dataset that drops.
- Compare the approval rate (`Loan_Status == 'Y'`) **among the rows that would be dropped** vs.
  **among the rows that would survive**. Are they the same?

This tells you whether the missingness is random (safe-ish to drop) or biased (dropping would
skew your training data). Print both rates.


### Step 4. Fix inconsistent casing

`Married`, `Self_Employed`, and `Property_Area` all contain mixed casing (`"yes"`, `"YES"`,
`"urban"`, ...). Strip whitespace and convert to title case for these three columns. Watch out —
`.str.title()` on a `NaN` produces the string `'Nan'`; convert that back to a real `np.nan`.

Print the sorted unique values of all three columns afterward to confirm they're clean.


### Step 5. Split BEFORE you impute

Build `X` from these columns:
```
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
```
and `y` as `Loan_Status == 'Y'` (as int).

Split into `X_train`, `X_test`, `y_train`, `y_test` with `test_size=0.2`, `random_state=42`,
`stratify=y`.

**Why split before imputing?** Write a one-line comment answering this in your own words.


### Step 6. Fix (impute) the numeric columns — don't drop them

Fit a `SimpleImputer(strategy='median')` on `X_train[num_cols]` only. Transform both
`X_train[num_cols]` and `X_test[num_cols]` with the fitted imputer (never re-fit on test).


### Step 7. Fix (impute) the categorical columns

Same idea, `SimpleImputer(strategy='most_frequent')`, fit on `X_train[cat_cols]` only, applied to
both splits.


### Step 8. Prove you fixed it instead of dropping it

Print:
- `X_train.isna().sum().sum()` and `X_test.isna().sum().sum()` — should both be `0`.
- `len(X_train)` and `len(X_test)` — compare them to the row counts right after Step 5's split.
  They should be **identical** (no rows lost).


### Step 9. One-hot encode

`pd.get_dummies` the categorical columns for train and test. Reindex the test matrix to match the
train matrix's columns (`fill_value=0`) so both have identical columns in identical order.


### Step 10 (stretch). Fixed data vs. dropped data — does it matter?

Train two `DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42)` models:

1. On your **fixed** `X_train_enc` / `y_train` from Step 9, evaluated on your fixed test set.
2. On a version built by `dropna()`-ing the raw `df` first, then splitting/encoding the same way.

Print both test accuracies and both test set sizes side by side. This is your evidence for why
fixing beats dropping — you keep more usable rows without sacrificing accuracy.
